## In this Tool call will be made based on LLM Response/Suggestions

In [1]:
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field, SecretStr
from pydantic import BaseModel, ValidationError
import json 
from functools import wraps
import numexpr as ne


class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_file="../.env")
    groq_api_key: SecretStr

In [2]:
settings = AppSettings()   # reads from .env / environment automatically
print(settings.groq_api_key)      

**********


### The expected structure from LLM
* City name along with weather information.
* Fahrenheit instead of Celsius.

# Langchain Style Tool Creation Decorator

In [10]:
import inspect
from typing import Callable, Any, Dict
from pydantic import create_model, BaseModel

class FuncationMetadataTool:
    """Wraps a Python function with metadata for an LLM."""
    def __init__(self, func: Callable, name: str, description: str, args_schema: type[BaseModel]):
        self.func = func
        self.name = name
        self.description = description
        self.args_schema = args_schema

    def __call__(self, *args, **kwargs) -> Any:
        # Validates arguments against the Pydantic schema before execution
        validated_args = self.args_schema(**kwargs)
        return self.func(**validated_args.model_dump())

    def get_llm_schema(self) -> Dict[str, Any]:
        """Generates OpenAI-style tool definition schema."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.args_schema.model_json_schema()
            }
        }

def tool(func: Callable) -> FuncationMetadataTool:
    """Decorator to transform a function into a CustomTool."""
    # Extract name and description
    name = func.__name__
    description = func.__doc__ or "No description provided."
    
    # Extract function signatures and type hints
    sig = inspect.signature(func)
    fields = {}
    
    for param_name, param in sig.parameters.items():
        if param_name == 'self':
            continue
        # Default to Any if no type hint is provided
        param_type = param.annotation if param.annotation != inspect.Parameter.empty else Any
        # Handle default values
        default_value = param.default if param.default != inspect.Parameter.empty else ...
        fields[param_name] = (param_type, default_value)
    
    # Dynamically create a Pydantic model for input validation
    schema_name = f"{name}"
    args_schema = create_model(schema_name, **fields)
    
    return FuncationMetadataTool(func, name, description, args_schema)


### Dummy Weather Tool, in reality information will be extracted from API's

In [11]:
@tool
def get_weather_information(city: str):
    """
    Retrieve weather information for a supported city.
    Args:
        city (str): The name of the city.
    Returns:
        dict: A dictionary containing:
            - celsius (int): Temperature in degrees Celsius.
            - conditions (str): A brief description of the weather.
    """
    weather = {
        "tokyo": {"celsius": 22, "conditions": "partly cloudy"},
        "delhi": {"celsius": 34, "conditions": "clear skies"},
        "london": {"celsius": 15, "conditions": "light rain"},
    }
    return weather.get(city.lower())

@tool
def calculator(expression: str) -> str:
    """
    Calculates mathematical expressions using numexpr.
    
    Args:
        expression: A string mathematical expression (e.g., "5.6 * (5 + 10.5)").
        
    Returns:
        The result of the calculation as a string.
    """
    try:
        result = ne.evaluate(expression)
        return f"The result of '{expression}' is {result}"
    except Exception as e:
        return f"Error evaluating expression: {e}"

In [12]:
print(json.dumps(calculator.get_llm_schema(), indent=2))

{
  "type": "function",
  "function": {
    "name": "calculator",
    "description": "\nCalculates mathematical expressions using numexpr.\n\nArgs:\n    expression: A string mathematical expression (e.g., \"5.6 * (5 + 10.5)\").\n\nReturns:\n    The result of the calculation as a string.\n",
    "parameters": {
      "properties": {
        "expression": {
          "title": "Expression",
          "type": "string"
        }
      },
      "required": [
        "expression"
      ],
      "title": "calculator",
      "type": "object"
    }
  }
}


### LLM with Tools Enabled
* Sends the question plus the tool schema in one call. 
* The reply may contain content or tool_calls list instead suggested by LLM.

In [ ]:
def ask_ai_to_choose(question: str):
    from openai import OpenAI
    client = OpenAI(api_key=settings.groq_api_key.get_secret_value(), base_url="https://api.groq.com/openai/v1")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        max_tokens=300,
        messages=[{"role": "user", "content": question}],
        tools=[get_weather_information.get_llm_schema(), calculator.get_llm_schema()],
    )
    return response.choices[0].message

<b> Asking question related to weather, we expect LLM to suggest ```get_weather_information``` tool call with parameter value.</b>
* message.content should be empty.
* message.tool_calls should have suggestion to call get_weather_information tool.

In [22]:
question = "Choose available tools when required and answer below question\n What is weather in delhi today"
message = ask_ai_to_choose(question)

print("Content: ",message.content)
if message.tool_calls:
    for i, tool_call in enumerate(message.tool_calls):
        print(f"Tool Call {i+1}: ",tool_call)

Content:  None
Tool Call 1:  ChatCompletionMessageFunctionToolCall(id='6kzn5tsyc', function=Function(arguments='{"city":"Delhi"}', name='get_weather_information'), type='function')


<b> Asking question related to maths equation, we expect LLM to suggest ```calculator``` tool call with parameter value.</b>
* message.content should be empty.
* message.tool_calls should have suggestion to call calculator tool.

In [21]:
question = "Choose available tools when required and answer below question\n Solve eqation (8+4)/6+4"
message = ask_ai_to_choose(question)

print("Content: ",message.content)
if message.tool_calls:
    for i, tool_call in enumerate(message.tool_calls):
        print(f"Tool Call {i+1}: ",tool_call)

Content:  None
Tool Call 1:  ChatCompletionMessageFunctionToolCall(id='yp4r73s5b', function=Function(arguments='{"expression":"(8+4)/6+4"}', name='calculator'), type='function')


<b> Asking generic question, we expect LLM to respond by itself.</b>
* message.content should have generated response.
* message.tool_calls should be empty

In [19]:
question = "Choose available tools when required and answer below question\n Hell llama! how are you"
message = ask_ai_to_choose(question)

print("Content: ",message.content)
print("Content: ",message.tool_calls)

Content:  I'm just an artificial intelligence language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to help with any questions or tasks you might have.
Content:  None


<b> Asking question related to weather and maths equation in same query<br>
We expect LLM to suggest 2 tool call ```calculator``` & ```get_weather_information```</b>
* message.content should be empty.
* message.tool_calls should have suggestion to call calculator tool.

In [23]:
question = """
    Choose available tools when required and answer below question
    Solve equation (8+4)/6+4 and also tell me What is weather in delhi today?
""".strip()
message = ask_ai_to_choose(question)

print("Content: ",message.content)
if message.tool_calls:
    for i, tool_call in enumerate(message.tool_calls):
        print(f"Tool Call {i+1}: ",tool_call)

Content:  None
Tool Call 1:  ChatCompletionMessageFunctionToolCall(id='yfbf52z7f', function=Function(arguments='{"expression":"(8+4)/6+4"}', name='calculator'), type='function')
Tool Call 2:  ChatCompletionMessageFunctionToolCall(id='q22j9pvkk', function=Function(arguments='{"city":"Delhi"}', name='get_weather_information'), type='function')
